In [1]:

import argparse
import json
import tqdm

import torch
import numpy
from transformers import AutoTokenizer, AutoModelForCausalLM


import os
import sys



import json
from typing import List

import torch
from torch.utils.data import Dataset

from prompt.templates.base import _PROMPT_PREFIX


def pprint_data(data: List, k: int = 3, n: int = 5):
    data = data[:k]
    for sample in data:
        id = sample.get('id')
        conversations = sample['input'].get('conversation')[:n]
        subject_keyword = sample['input'].get('subject_keyword')
        outputs = sample['output']
        print(f"\n**** Conversation ID: {id} ****\n")
        print("INPUTS:")
        print("\t<대화내용>")
        for cvt in conversations:
            speaker = cvt.get('speaker')
            utterance = cvt.get('utterance', '')
            print(f'\t- 화자({speaker}): {utterance}')
        print('\t- ...')
        print(f'\t<핵심 키워드>: {subject_keyword[0]}\n')
        print(f"OUTPUT: {outputs}\n\n")


class CustomDataset(Dataset):
    def __init__(self, fname, tokenizer):
        IGNORE_INDEX = -100
        self.inp = []
        self.label = []

        PROMPT = _PROMPT_PREFIX

        with open(fname, "r") as f:
            data = json.load(f)

        def make_chat(inp):
            chat = ["[Conversation]"]
            for cvt in inp['conversation']:
                speaker = cvt['speaker']
                utterance = cvt['utterance']
                chat.append(f"화자{speaker}: {utterance}")
            chat = "\n".join(chat)
            keyword = ', '.join(inp['subject_keyword'])
            question = f"[Question]\n위 {keyword} 주제에 대한 대화를 요약해주세요."
            chat = chat + "\n\n" + question

            return chat

        for example in data:
            chat = make_chat(example["input"])
            message = [
                {"role": "system", "content": PROMPT},
                {"role": "user", "content": chat},
            ]

            source = tokenizer.apply_chat_template(
                message,
                add_generation_prompt=True,
                return_tensors="pt",
            )

            target = example["output"]
            if target != "":
                target += tokenizer.eos_token
            target = tokenizer(target,
                               return_attention_mask=False,
                               add_special_tokens=False,
                               return_tensors="pt")
            target["input_ids"] = target["input_ids"].type(torch.int64)

            input_ids = torch.concat((source[0], target["input_ids"][0]))
            labels = torch.concat((torch.LongTensor([IGNORE_INDEX] * source[0].shape[0]), target["input_ids"][0]))
            self.inp.append(input_ids)
            self.label.append(labels)

    def __len__(self):
        return len(self.inp)

    def __getitem__(self, idx):
        return self.inp[idx]


class DataCollatorForSupervisedDataset(object):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, instances):
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(ids) for ids in input_ids], batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence([torch.tensor(lbls) for lbls in labels], batch_first=True,
                                                 padding_value=-100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )





class Config:
    output = '/data1/kaggle/jok-llmarchi-kli/resource/outputs/'
    model_id = "/data1/kaggle/jok-llmarchi-kli/finetuned_model/batch_size(16)_epoch(10)_peft_method(lora)_r(8)_alpha(16)_target_modules({'gate_proj', 'down_proj', 'v_proj', 'o_proj', 'k_proj', 'up_proj', 'q_proj'})_load_bit(4bit)_compute_type(torch.float32)/checkpoint-126(1.42)"
    tokenizer = model_id
    device = 'cuda'

args = Config

from transformers import AutoModelForCausalLM
from peft import LoraConfig, PeftModel

# LoRA 설정 불러오기
lora_config = LoraConfig.from_pretrained(args.model_id)

# 기본 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    "MLP-KTLim/llama-3-Korean-Bllossom-8B",
    # torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir="./models/",
    torch_dtype=torch.bfloat16
)

# LoRA 가중치 로드
model = PeftModel.from_pretrained(model, args.model_id)
model.eval()


/home/rainism/anaconda3/envs/ko_finetuning/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:39<00:00,  9.86s/it]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=256, out_features=256, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=256, out_features=256, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear(
                (base_layer): Linear(in_features

In [2]:

if args.tokenizer == None:
    args.tokenizer = args.model_id
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer)
tokenizer.pad_token = tokenizer.eos_token
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

dataset = CustomDataset("../baseline/resource/data/일상대화요약_test.json", tokenizer)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:


with open("../baseline/resource/data/일상대화요약_test.json", "r") as f:
    result = json.load(f)

for idx in tqdm.tqdm(range(len(dataset))):
    inp = dataset[idx]
    outputs = model.generate(
        inp.to(args.device).unsqueeze(0),
        max_new_tokens=1000,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False,
    )

    result[idx]["output"] = tokenizer.decode(outputs[0][inp.shape[-1]:], skip_special_tokens=True)


  0%|          | 0/408 [00:00<?, ?it/s]

/home/rainism/anaconda3/envs/ko_finetuning/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/rainism/anaconda3/envs/ko_finetuning/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
100%|██████████| 408/408 [2:16:04<00:00, 20.01s/it]  


In [4]:

with open(args.output + '일상대화요약_test_submit(mora(8,16),qkvoj,quantization).json', "w", encoding="utf-8") as f:
    f.write(json.dumps(result, ensure_ascii=False, indent=4))
